In [1]:
"""Procesamiento de datos con segmentacion y ruido.

Esta funcion lee los archivos originales, los divide en segmentos,
introduce ruido gaussiano y devuelve un DataFrame con una unica
observacion por intervalo de tiempo.
"""

from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd


def preparar_datos(
    ruido: float,
    frecuencia_observacion: float,
    periodo_orbital: float,
    numero_segmentos: int,
) -> pd.DataFrame:
    """Devuelve un DataFrame con una observacion por intervalo."""

    raw_files = [
        "data/original/mdot_series_q010_e000.txt",
        "data/original/mdot_series_q025_e000.txt",
        "data/original/mdot_series_q050_e000.txt",
        "data/original/mdot_series_q100_e000.txt",
        "data/original/mdot_series_q100_e050.txt",
        "data/original/mdot_series_q100_e040.txt",
        "data/original/mdot_series_q100_e030.txt",
        "data/original/mdot_series_q100_e020.txt",
        "data/original/mdot_series_q100_e010.txt",
        "data/original/mdot_series_q100_e060.txt",
        "data/original/mdot_series_q100_e070.txt",
    ]

    segments_dir = Path("data/segments")
    segments_dir.mkdir(parents=True, exist_ok=True)

    float_fmt = "%.18e"
    for raw in raw_files:
        raw_path = Path(raw)
        if not raw_path.exists():
            continue
        df_raw = pd.read_csv(raw_path, delim_whitespace=True, header=None)
        rows_per_seg = len(df_raw) // numero_segmentos
        for i in range(numero_segmentos):
            start = i * rows_per_seg
            end = (i + 1) * rows_per_seg if i < numero_segmentos - 1 else len(df_raw)
            seg_df = df_raw.iloc[start:end]
            seg_name = raw_path.stem + f"_seg{i+1:02d}" + raw_path.suffix
            seg_path = segments_dir / seg_name
            if not seg_path.exists():
                seg_df.to_csv(seg_path, sep=" ", header=False, index=False, float_format=float_fmt)

    pattern = "*_seg??.txt"
    col_time = "time"
    col_primary = "acre_rate_primary"
    col_second = "acre_rate_secondary"
    col_mw = "acre_rate_mass_weighted"

    def _parse(fname: str) -> tuple[float | None, float | None, int | None]:
        m = re.search(r"q(\d+)_e(\d+)_seg(\d+)", fname)
        if not m:
            return None, None, None
        q = int(m.group(1)) / 100.0
        e = int(m.group(2)) / 100.0
        s = int(m.group(3))
        return q, e, s

    frames: list[pd.DataFrame] = []
    for f in sorted(segments_dir.glob(pattern)):
        q_val, e_val, seg_id = _parse(f.name)
        if q_val is None:
            continue
        df = pd.read_csv(f, delim_whitespace=True, header=None, names=[col_time, col_primary, col_second])
        df["q"], df["e"], df["seg"] = q_val, e_val, seg_id
        w1 = 1 / (1 + q_val)
        w2 = q_val / (1 + q_val)
        df[col_mw] = w1 * df[col_primary].abs() + w2 * df[col_second].abs()
        q_tag = f"{int(round(q_val * 100)):03d}"
        e_tag = f"{int(round(e_val * 100)):03d}"
        seg_tag = f"{seg_id:02d}"
        df["id"] = f"{q_tag}_{e_tag}_{seg_tag}"
        frames.append(df)

    all_data = pd.concat(frames, ignore_index=True)
    all_data = all_data[~all_data["seg"].isin({1, 2})]

    rng = np.random.default_rng()
    sigma = all_data[col_mw].std(ddof=1)
    all_data[f"{col_mw}_clean"] = all_data[col_mw]
    noise = rng.normal(0.0, ruido * sigma, size=len(all_data))
    all_data[col_mw] = all_data[col_mw] + noise

    all_data["time_days"] = all_data[col_time] * periodo_orbital
    all_data["bucket"] = np.floor((all_data["time_days"] + 1e-9) / frecuencia_observacion).astype(int)

    unique_df = (
        all_data
        .groupby(["id", "bucket"], as_index=False)
        .median(numeric_only=True)
        .sort_values(["id", "time_days"])
        .reset_index(drop=True)
        .drop(columns="bucket")
    )

    return unique_df

In [2]:
"""Analisis de similitud basado en el cuaderno analisis-RF."""
from __future__ import annotations

import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from pycatch22 import catch22_all


def _extract_catch22(id_: str, grp: pd.DataFrame) -> pd.Series:
    values = grp.sort_values("time")["acre_rate_mass_weighted"].to_numpy()
    res = catch22_all(values)
    return pd.Series(res["values"], index=res["names"], name=id_)


def _compute_features(df: pd.DataFrame, n_jobs: int = -1) -> pd.DataFrame:
    grouped = df.groupby("id", sort=False)
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_extract_catch22)(i, g) for i, g in grouped
    )
    features_df = pd.concat(results, axis=1).T
    features_df.reset_index(inplace=True)
    features_df.rename(columns={"index": "id"}, inplace=True)
    return features_df


def _standardize_features(df: pd.DataFrame) -> tuple[pd.DataFrame, StandardScaler]:
    feature_cols = df.columns.difference(["id"])
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df[feature_cols])
    df_std = pd.DataFrame(scaled, columns=feature_cols, index=df.index)
    df_std.insert(0, "id", df["id"])
    return df_std, scaler


def _weight_features(df_std: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    df_std[["q", "e", "seg"]] = df_std["id"].str.split("_", expand=True)
    df_std["q"] = df_std["q"].astype(int) / 100
    df_std["e"] = df_std["e"].astype(int) / 100
    df_std.drop(columns="seg", inplace=True)

    id_col = "id"
    target_cols = ["q", "e"]
    feature_cols = df_std.columns.difference([id_col] + target_cols)

    Y_enc = pd.DataFrame(index=df_std.index)
    for col in target_cols:
        le = LabelEncoder().fit(df_std[col])
        Y_enc[col] = le.transform(df_std[col])

    X = df_std[feature_cols]
    Y = Y_enc[target_cols]

    base = RandomForestClassifier(random_state=42, n_jobs=-1)
    multi = MultiOutputClassifier(base, n_jobs=-1)
    grid = GridSearchCV(
        multi,
        {
            "estimator__n_estimators": [200, 500],
            "estimator__max_depth": [None, 20, 40],
            "estimator__min_samples_leaf": [1, 2, 4],
            "estimator__max_features": ["sqrt", "log2"],
        },
        cv=5,
        scoring="r2",
        n_jobs=-1,
    )
    best = grid.fit(X, Y).best_estimator_
    importances = np.vstack([est.feature_importances_ for est in best.estimators_])
    importances = importances.mean(axis=0)
    importances /= importances.sum()

    df_weighted = df_std.copy()
    df_weighted[feature_cols] = df_weighted[feature_cols].mul(importances, axis=1)
    return df_weighted, importances


def evaluar_similitud(
    df: pd.DataFrame,
    TARGET: str,
    id_sim: str,
    distancia: str = "manhattan",
) -> bool:
    """Evalua si `id_sim` es la serie mas cercana al TARGET."""
    features = _compute_features(df)
    df_std, _ = _standardize_features(features)
    df_w, _ = _weight_features(df_std)

    feature_cols = df_w.columns.difference(["id", "q", "e"])

    x0 = df_w.loc[df_w["id"] == TARGET, feature_cols].to_numpy()
    X = df_w.loc[df_w["id"] != TARGET, feature_cols].to_numpy()

    metric = distancia
    kwargs = {}
    if distancia == "manhattan":
        metric = "cityblock"
    elif distancia == "mahalanobis":
        cov = np.cov(df_w[feature_cols].to_numpy(), rowvar=False)
        kwargs["VI"] = np.linalg.inv(cov)

    dist = cdist(X, x0, metric=metric, **kwargs).flatten()
    other_ids = df_w.loc[df_w["id"] != TARGET, "id"].to_numpy()
    dist_df = pd.DataFrame({"id": other_ids, distancia: dist})

    dist_df[["q", "e", "seg"]] = dist_df["id"].str.split("_", expand=True)
    dist_df["q"] = dist_df["q"].astype(int) / 100
    dist_df["e"] = dist_df["e"].astype(int) / 100
    dist_df.drop(columns="seg", inplace=True)

    scaler = MinMaxScaler(feature_range=(0, 1))
    dist_df[[distancia]] = scaler.fit_transform(dist_df[[distancia]])

    median = (
        dist_df
        .groupby(["q", "e"], as_index=False)[distancia]
        .median()
        .sort_values(distancia)
    )
    median["id"] = median.apply(
        lambda r: f"{int(round(r['q'] * 100)):03d}_{int(round(r['e'] * 100)):03d}",
        axis=1,
    )
    best_id = median.iloc[0]["id"]
    return id_sim == best_id

In [3]:
from itertools import product
import pandas as pd
import numpy as np

# ───────────────────────────────────────────────────────
# 1. Pequeño helper para incluir el límite superior
# ───────────────────────────────────────────────────────
def r(min_, max_, step):
    n = int(round((max_ - min_) / step))
    return [min_ + i * step for i in range(n + 1)]

# ───────────────────────────────────────────────────────
# 2. Bucle mínimo: un solo id_sim fijo
# ───────────────────────────────────────────────────────
def correr_grid(
    preparar_datos_fn,
    evaluar_similitud_fn,
    ruido_rg, frec_rg, per_rg, nseg_rg,
    target_id,           # «TARGET» para evaluar_similitud
    id_sim,              # id_sim que quieres testar
    distancia="manhattan",
):
    filas = []
    fallidos = []

    for ruido, frec, per, nseg in product(r(*ruido_rg),
                                          r(*frec_rg),
                                          r(*per_rg),
                                          r(*nseg_rg)):
        df = preparar_datos_fn(ruido, frec, per, int(nseg))
        try:
            ok = evaluar_similitud_fn(
                df,
                TARGET=target_id,
                id_sim=id_sim,
                distancia=distancia
            )
            filas.append((ruido, frec, per, int(nseg), ok))
        except ValueError as exc:
            if "NaN" in str(exc) or "Input contains NaN" in str(exc):
                fallidos.append((ruido, frec, per, int(nseg)))
            else:
                raise

    return pd.DataFrame(
        filas,
        columns=["ruido", "frec_obs", "periodo", "n_seg", "ok"]
    ), fallidos


In [4]:
import time
_t0 = time.perf_counter()


In [5]:
ruido_rg = (0.1,10, 0.1)
frec_rg  = (1, 28, 7 )
per_rg   = (1, 330, 30) #30-300-100
nseg_rg  = (20,20, 1)

df_res = correr_grid(
    preparar_datos_fn   = preparar_datos,
    evaluar_similitud_fn= evaluar_similitud,
    ruido_rg = ruido_rg,
    frec_rg  = frec_rg,
    per_rg   = per_rg,
    nseg_rg  = nseg_rg,
    target_id= "100_050_20",   # tu TARGET
    id_sim   = "100_050"    # el id_sim que quieras comprobar
)



In [7]:
print(f"⏱️  Notebook completo: {time.perf_counter()-_t0:,.2f} s")


⏱️  Notebook completo: 77,699.68 s


In [17]:
df_only = df_res[0]


In [20]:
df_only.to_csv("data/sensibilidad.csv")
